# Preparazione testi per embeddings

Questo notebook crea le rappresentazioni JSONL da usare nel prossimo step di invoice matching semantico.

L'obiettivo e' separare:
- `embedding_text`: solo contenuto testuale e semantico utile al retrieval vettoriale;
- `metadata`: campi strutturati, numerici o identificativi da usare dopo, per filtri SAP, controlli esatti, fuzzy matching o reranking.

Non vengono chiamati modelli embedding, vector database o API FastAPI.

## 1. Import e percorsi

In [1]:
import json
from pathlib import Path
from typing import Any


current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
DATA_DIR = PROJECT_ROOT / "data" / "processed"

GROUND_TRUTH_INPUT = DATA_DIR / "invoices_ground_truth_corpus.jsonl"
NOISY_EVALUATION_INPUT = DATA_DIR / "invoices_noisy_evaluation.jsonl"
EMBEDDING_CORPUS_OUTPUT = DATA_DIR / "embedding_corpus.jsonl"
EMBEDDING_QUERIES_OUTPUT = DATA_DIR / "embedding_queries.jsonl"

GROUND_TRUTH_INPUT, NOISY_EVALUATION_INPUT, EMBEDDING_CORPUS_OUTPUT, EMBEDDING_QUERIES_OUTPUT

(WindowsPath('C:/Users/pietr/Documents/ai-invoice-matching/ai-invoice-matching/data/processed/invoices_ground_truth_corpus.jsonl'), WindowsPath('C:/Users/pietr/Documents/ai-invoice-matching/ai-invoice-matching/data/processed/invoices_noisy_evaluation.jsonl'), WindowsPath('C:/Users/pietr/Documents/ai-invoice-matching/ai-invoice-matching/data/processed/embedding_corpus.jsonl'), WindowsPath('C:/Users/pietr/Documents/ai-invoice-matching/ai-invoice-matching/data/processed/embedding_queries.jsonl'))

## 2. Funzioni di utilita'

In [2]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON non valido in {path} alla riga {line_number}") from exc
    return records


def write_jsonl(path: Path, records: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True) + "\n")


def present(value: Any) -> bool:
    return value is not None and value != ""


def without_keys(data: dict[str, Any], excluded_keys: set[str]) -> dict[str, Any]:
    return {key: value for key, value in data.items() if key not in excluded_keys and present(value)}

## 3. Separazione tra testo semantico e metadata

`embedding_text` contiene solo:
- seller company name;
- buyer company name;
- seller address;
- buyer address;
- buyer country;
- product descriptions.

Invoice number, date, quantita', prezzi, totali, sconti, VAT number e altri campi strutturati restano in `metadata`.

In [3]:
def build_embedding_text(invoice: dict[str, Any]) -> str:
    seller = invoice.get("seller") or {}
    buyer = invoice.get("buyer") or {}
    products = invoice.get("products") or []

    lines = []
    semantic_fields = [
        ("Seller company", seller.get("company_name")),
        ("Buyer company", buyer.get("company_name")),
        ("Seller address", seller.get("address")),
        ("Buyer address", buyer.get("address")),
        ("Buyer country", buyer.get("country")),
    ]

    for label, value in semantic_fields:
        if present(value):
            lines.append(f"{label}: {value}")

    descriptions = [product.get("description") for product in products if isinstance(product, dict) and present(product.get("description"))]
    if descriptions:
        lines.append("Product descriptions:")
        lines.extend(f"- {description}" for description in descriptions)

    return "\n".join(lines)


def build_invoice_metadata(invoice: dict[str, Any]) -> dict[str, Any]:
    metadata: dict[str, Any] = {}

    seller_metadata = without_keys(invoice.get("seller") or {}, {"company_name", "address"})
    buyer_metadata = without_keys(invoice.get("buyer") or {}, {"company_name", "address", "country"})

    if seller_metadata:
        metadata["seller"] = seller_metadata
    if buyer_metadata:
        metadata["buyer"] = buyer_metadata

    for section in ["invoice", "payment", "tax", "supplier"]:
        value = invoice.get(section)
        if isinstance(value, dict):
            cleaned = {key: item for key, item in value.items() if present(item)}
            if cleaned:
                metadata[section] = cleaned
        elif present(value):
            metadata[section] = value

    product_metadata = []
    for product in invoice.get("products") or []:
        if not isinstance(product, dict):
            continue
        cleaned_product = without_keys(product, {"description"})
        if cleaned_product:
            product_metadata.append(cleaned_product)
    if product_metadata:
        metadata["products"] = product_metadata

    return metadata

## 4. Creazione dei record per corpus e query

In [4]:
def build_embedding_record(source_record: dict[str, Any], invoice_key: str, extra_metadata_keys: list[str] | None = None) -> dict[str, Any]:
    invoice = source_record[invoice_key]
    metadata = build_invoice_metadata(invoice)

    for key in extra_metadata_keys or []:
        if key in source_record and present(source_record[key]):
            metadata[key] = source_record[key]

    return {
        "record_id": source_record["record_id"],
        "source_image": source_record.get("source_image"),
        "embedding_text": build_embedding_text(invoice),
        "metadata": metadata,
    }


ground_truth_records = read_jsonl(GROUND_TRUTH_INPUT)
noisy_evaluation_records = read_jsonl(NOISY_EVALUATION_INPUT)

embedding_corpus = [
    build_embedding_record(record, invoice_key="ground_truth")
    for record in ground_truth_records
]

embedding_queries = [
    build_embedding_record(
        record,
        invoice_key="noisy_invoice",
        extra_metadata_keys=["evaluation_split", "noise_level", "error_types", "changed_fields"],
    )
    for record in noisy_evaluation_records
]

len(embedding_corpus), len(embedding_queries)

(7000, 2100)

## 5. Scrittura dei file JSONL

In [5]:
write_jsonl(EMBEDDING_CORPUS_OUTPUT, embedding_corpus)
write_jsonl(EMBEDDING_QUERIES_OUTPUT, embedding_queries)

{
    "embedding_corpus_path": str(EMBEDDING_CORPUS_OUTPUT),
    "embedding_corpus_records": len(embedding_corpus),
    "embedding_queries_path": str(EMBEDDING_QUERIES_OUTPUT),
    "embedding_queries_records": len(embedding_queries),
}

{'embedding_corpus_path': 'C:\\Users\\pietr\\Documents\\ai-invoice-matching\\ai-invoice-matching\\data\\processed\\embedding_corpus.jsonl', 'embedding_corpus_records': 7000, 'embedding_queries_path': 'C:\\Users\\pietr\\Documents\\ai-invoice-matching\\ai-invoice-matching\\data\\processed\\embedding_queries.jsonl', 'embedding_queries_records': 2100}

## 6. Controlli rapidi

In [6]:
embedding_corpus[0]

{'record_id': 1, 'source_image': '001.png', 'embedding_text': 'Seller company: Lopez, Miller and Romero\nBuyer company: Hayes LLC\nSeller address: 60464 Curtis Gateway East Keith, IN 57123\nBuyer address: 960 Hurley Springs North Alyssa, RI 49322\nBuyer country: Lebanon\nProduct descriptions:\n- transform open-source info-mediaries\n- innovate transparent e-services\n- optimize innovative action-items\n- enhance e-business initiatives\n- morph value-added markets\n- cultivate compelling systems\n- re-contextualize leading-edge communities', 'metadata': {'seller': {'vat_no': 'Qu96730920377'}, 'buyer': {'name': 'Mercedes Martinez'}, 'invoice': {'date': '05.08.2007', 'number': 802205}, 'payment': {'discount_amount': 43.03, 'discount_percentage': 11.77, 'due': 61.79, 'sub_total': 141.66, 'total': 534.11}, 'products': [{'index': '10', 'quantity': 6.27, 'total_price': 329.62, 'unit_price': 41.57}, {'index': '9', 'quantity': 5.62, 'total_price': 986.28, 'unit_price': 10.47}, {'index': '3', 'q

In [7]:
embedding_queries[0]

{'record_id': 4, 'source_image': '004.png', 'embedding_text': "Seller company: Holt-Mercado\nBuyer company: Barnett, White and James\nSeller address: 45838 Stephens Stream East Jessica, KY 44332\nBuyer address: 44126 Claudia Mount Suite 784 Morrisfort, IA 95132\nBuyer country: Lao People's Democratic Republic\nProduct descriptions:\n- embrace wireless architectures", 'metadata': {'seller': {'vat_no': 'kz32957163737'}, 'buyer': {'name': 'Nicole Martin'}, 'invoice': {'date': '02.10.1993', 'number': 807426}, 'payment': {'discount_amount': 23.88, 'discount_percentage': 8.1, 'due': 89.63, 'sub_total': 64.0, 'total': 704.26}, 'products': [{'index': '10', 'quantity': 4.31, 'total_price': 927.22, 'unit_price': 42.13}], 'evaluation_split': 'test', 'noise_level': 'low', 'error_types': ['decimal_rounding'], 'changed_fields': ['payment.sub_total']}}

In [8]:
query_splits = {}
for record in embedding_queries:
    split = record["metadata"].get("evaluation_split")
    query_splits[split] = query_splits.get(split, 0) + 1

query_splits

{'test': 1050, 'validation': 1050}